In [ ]:

%load_ext autoreload
%autoreload 2

from itertools import product
import sys
sys.path.append('/home/projects/nyosef/zvise/PixelGen/PixelGen/')

import anndata as ad
import pixelator
import torch
import scvi
import scipy

import seaborn as sns
import scanpy as sc
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from tqdm import tqdm

from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection
import tempfile

scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)
sc.set_figure_params(figsize=(6, 6), frameon=False)

sns.set_theme()
torch.set_float32_matmul_precision("high")
save_dir = tempfile.TemporaryDirectory()

from utils import run_marker_community_analysis, plot_latent, plot_gene_heatmap, plot_model_latents,get_dense,calculate_metrics,plot_composite_ppc, run_de_analysis, run_pseudobulk_pipeline
%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"
MODEL_PATH='/home/projects/nyosef/zvise/PixelGen/PixelGen/PBMSC/models'
SEED = 30
import random
from sklearn.preprocessing import StandardScaler

np.random.seed(SEED)
random.seed(SEED)

import networkx as nx

In [ ]:
adata.obs.cell_type.value_counts()

In [ ]:
adata=sc.read_h5ad('/home/projects/nyosef/zvise/PixelGen/PixelGen/PBMSC/cache/pbmsc_adata_final_annotated.h5ad')
cd8= adata[adata.obs['cell_type']=='Monocytes'].copy()
transformed_matrix = np.arcsinh(adata.obsm['HOTSPOT_top500_var'] * 100)
adata.obsm['hotspot_500_arcsinh'] = transformed_matrix

# SIMPLE DE

In [ ]:
diff_exp_df = run_de_analysis(
    adata=cd8, 
    condition='condition', 
    test_group='PHA', 
    ref_group='unstim', 
    cell_filter="cell_type == 'CD8'",  # Replaces the old cell_type argument
    layer='arcsinh', 
)

# Pseudobulk Differential Expression Analysis

Perform pseudobulk DE analysis for:
- **Abundance**: Protein expression levels
- **Hotspot**: Spatial colocalization features (Moran's I based)
- **Joint count**: Proximity-based colocalization features

Compare PHA vs unstimulated samples.

In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt

def run_obsm_de_analysis(adata, condition='condition', test_group='PHA', ref_group='unstim', n_label=5):
    """
    Runs DE analysis specifically on .obsm matrices containing '500' in their name.
    Creates temporary AnnData objects for each matrix to allow RankGenesGroups to work.
    """
    
    # 1. Identify relevant obsm keys (Strict filter: must contain "500")
    target_keys = [k for k in adata.obsm.keys() if '500' in k]
    
    if not target_keys:
        print("No .obsm keys found containing '500'.")
        return

    print(f"Found {len(target_keys)} matrices to analyze: {target_keys}")

    for key in target_keys:
        print(f"\n{'='*80}\nANALYZING: {key}\n{'='*80}")
        
        # 2. Create Temporary AnnData
        modality_data = pd.DataFrame(adata.obsm[key], index=adata.obs_names)
        
        # Clean column names if integers
        if isinstance(modality_data.columns[0], int):
             modality_data.columns = [f"Module_{i}" for i in modality_data.columns]
             
        t_adata = sc.AnnData(modality_data)
        t_adata.obs = adata.obs.copy() 
        
        # 3. Run Differential Expression
        try:
            sc.tl.rank_genes_groups(t_adata, groupby=condition, groups=[test_group], 
                                    reference=ref_group, method='wilcoxon')
        except Exception as e:
            print(f"Skipping {key}: Could not run DE stats ({e})")
            continue
            
        df = sc.get.rank_genes_groups_df(t_adata, group=test_group).copy()
        
        # 4. Metrics
        df['nlog10'] = -np.log10(df['pvals_adj'] + 1e-300)
        df['color'] = np.where((df.pvals_adj < 0.05) & (df.logfoldchanges > 0.5), f'Up in {test_group}',
                      np.where((df.pvals_adj < 0.05) & (df.logfoldchanges < -0.5), f'Up in {ref_group}', 'NS'))
        
        # Calculate Prevalence
        mask = t_adata.obs[condition] == test_group
        raw_data = t_adata.X[mask]
        if hasattr(raw_data, "toarray"): raw_data = raw_data.toarray()
            
        mean_val = (raw_data > 0).mean(axis=0) * 100
        pct_map = dict(zip(t_adata.var_names, mean_val))
        df['pct'] = df['names'].map(pct_map)

        # 5. Visualization A: Volcano Plot
        plt.figure(figsize=(8, 6))
        sns.scatterplot(data=df, x='logfoldchanges', y='nlog10', hue='color', size='pct', 
                        palette={f'Up in {test_group}':'#d62728', f'Up in {ref_group}':'#1f77b4', 'NS':'lightgrey'}, 
                        sizes=(10, 200), alpha=0.6)

        top_hits = pd.concat([df.nlargest(n_label, 'logfoldchanges'), df.nsmallest(n_label, 'logfoldchanges')])
        for _, r in top_hits.iterrows():
            plt.text(r['logfoldchanges'], r['nlog10'], r['names'], fontsize=9, weight='bold')
        
        plt.title(f'Differential Analysis: {key}\n{test_group} vs {ref_group}')
        sns.despine(); plt.show()

        # 6. Visualization B: Heatmap (Top 20 DE Features)
        top_features = df.nlargest(20, 'logfoldchanges')['names'].tolist()
        if len(top_features) > 0:
            sc.pl.heatmap(t_adata, var_names=top_features, groupby=condition, 
                          standard_scale='var', cmap='RdBu_r', swap_axes=True, show=True)

        # 7. Print Tables (Styled Top 10 Up and Down)
        top_up = df[df['logfoldchanges'] > 0].nlargest(10, 'logfoldchanges')
        top_down = df[df['logfoldchanges'] < 0].nsmallest(10, 'logfoldchanges')
        
        print(f"\n{'='*40}\nTOP 10 UPREGULATED IN {test_group} (PHA) - {key}\n{'='*40}")
        display(top_up[['names', 'logfoldchanges', 'pvals_adj', 'pct']].style.background_gradient(cmap='Reds', subset=['logfoldchanges']))
        
        print(f"\n{'='*40}\nTOP 10 UPREGULATED IN {ref_group} (Unstim) - {key}\n{'='*40}")
        display(top_down[['names', 'logfoldchanges', 'pvals_adj', 'pct']].style.background_gradient(cmap='Blues_r', subset=['logfoldchanges']))

# Run the analysis on the 'cd8' object
run_obsm_de_analysis(cd8)

# Pseudobulk Differential Expression Analysis

Perform pseudobulk DE analysis for:
- **Abundance**: Protein expression levels
- **Hotspot**: Spatial colocalization features (Moran's I based)
- **Joint count**: Proximity-based colocalization features

Compare PHA vs unstimulated samples.

In [ ]:
run_pseudobulk_pipeline(cd8)

# Task 4: Marker Community Detection

Find groups of markers that are close together (colocalized), not just pairs.

Uses graph-based community detection:
1. Build graph where nodes = markers, edges = high colocalization pairs
2. Apply Louvain/Leiden community detection to find marker modules
3. Visualize and interpret the marker communities

In [ ]:
# Per-sample highly variable marker selection (rank-based pooling)
# For each sample, compute variance of each marker's arcsinh expression,
# rank markers within each sample, then average ranks across samples.
# Lower mean rank = more consistently highly variable.

layer_key = 'arcsinh'
sample_col = 'sample'

expr = cd8.to_df(layer_key)
samples = cd8.obs[sample_col]

# Variance per marker per sample, then rank within each sample
var_per_sample = expr.groupby(samples).var()                    # (n_samples x n_markers)
rank_per_sample = var_per_sample.rank(axis=1, ascending=False)  # rank 1 = highest variance

# Pool: mean rank across samples
mean_rank = rank_per_sample.mean(axis=0).sort_values()

# Summary dataframe for plotting and selection
hvg_df = pd.DataFrame({
    'marker': mean_rank.index,
    'mean_rank': mean_rank.values,
    'mean_variance': var_per_sample.mean(axis=0).loc[mean_rank.index].values,
    'n_samples_top50pct': (rank_per_sample <= rank_per_sample.shape[1] / 2).sum(axis=0).loc[mean_rank.index].values,
})

print(f"Total markers: {len(hvg_df)}")
print(f"Samples: {var_per_sample.shape[0]}")
print(f"\nTop 10 most consistently variable markers:")
display(hvg_df.head(10))

In [ ]:
n_top = 100  # adjustable — look at plots below to decide

fig, axes = plt.subplots(2, 2, figsize=(22, 16))

# --- Plot A: Per-sample variance heatmap (sorted by mean rank) ---
ax = axes[0, 0]
sorted_var = var_per_sample[mean_rank.index]  # columns sorted by mean rank
sns.heatmap(sorted_var, cmap='viridis', ax=ax, cbar_kws={"shrink": 0.6, "label": "Variance"})
ax.axvline(x=n_top, color='red', linewidth=2, linestyle='--', label=f'Top {n_top} cutoff')
ax.set_title('A. Per-Sample Marker Variance (sorted by mean rank)')
ax.set_xlabel('Markers (sorted by mean rank →)')
ax.set_ylabel('Sample')
# Only show every Nth tick to avoid crowding
tick_step = max(1, len(mean_rank) // 30)
ax.set_xticks(range(0, len(mean_rank), tick_step))
ax.set_xticklabels([mean_rank.index[i] for i in range(0, len(mean_rank), tick_step)], rotation=90, fontsize=7)
ax.legend(loc='upper right')

# --- Plot B: Mean rank barplot with cutoff ---
ax = axes[0, 1]
colors = ['#d62728' if i < n_top else '#aec7e8' for i in range(len(hvg_df))]
ax.barh(range(len(hvg_df)), hvg_df['mean_rank'].values, color=colors, edgecolor='none')
ax.axhline(y=n_top - 0.5, color='black', linewidth=2, linestyle='--', label=f'Top {n_top} cutoff')
ax.set_ylabel('Markers (sorted by mean rank)')
ax.set_xlabel('Mean Rank Across Samples')
ax.set_title(f'B. Mean Rank of Markers (red = selected top {n_top})')
ax.set_yticks(range(0, len(hvg_df), max(1, len(hvg_df) // 30)))
ax.set_yticklabels([hvg_df.iloc[i]['marker'] for i in range(0, len(hvg_df), max(1, len(hvg_df) // 30))], fontsize=7)
ax.invert_yaxis()
ax.legend(loc='lower right')

# --- Plot C: Rank consistency scatter ---
ax = axes[1, 0]
n_samples = var_per_sample.shape[0]
sc_plot = ax.scatter(hvg_df['mean_variance'], hvg_df['mean_rank'],
                     c=hvg_df['n_samples_top50pct'], cmap='RdYlGn', s=60, edgecolors='black', linewidths=0.3,
                     vmin=0, vmax=n_samples)
plt.colorbar(sc_plot, ax=ax, label=f'# samples in top 50% (of {n_samples})', shrink=0.7)
ax.axhline(y=hvg_df.iloc[n_top - 1]['mean_rank'], color='red', linestyle='--', alpha=0.5, label=f'Top {n_top} cutoff')

# Label top markers and some borderline ones
for _, row in pd.concat([hvg_df.head(10), hvg_df.iloc[n_top-3:n_top+3]]).drop_duplicates().iterrows():
    ax.annotate(row['marker'], (row['mean_variance'], row['mean_rank']), fontsize=7, alpha=0.8)

ax.set_xlabel('Mean Variance Across Samples')
ax.set_ylabel('Mean Rank (lower = more variable)')
ax.set_title('C. Variance vs Rank (color = consistency)')
ax.legend(loc='lower right')

# --- Plot D: Selected vs excluded variance distributions ---
ax = axes[1, 1]
selected_set = set(hvg_df.head(n_top)['marker'])

# Melt variance data for violin plot
var_melted = var_per_sample.melt(var_name='marker', value_name='variance', ignore_index=False)
var_melted['group'] = var_melted['marker'].apply(lambda x: f'Selected (top {n_top})' if x in selected_set else 'Excluded')

sns.violinplot(data=var_melted, x='group', y='variance', ax=ax, inner='box', cut=0,
               palette={f'Selected (top {n_top})': '#d62728', 'Excluded': '#aec7e8'})
ax.set_title('D. Variance Distribution: Selected vs Excluded Markers')
ax.set_xlabel('')
ax.set_ylabel('Variance (arcsinh layer)')

plt.tight_layout()
plt.suptitle(f'Highly Variable Marker Selection (n_top = {n_top})', fontsize=14, y=1.01)
plt.show()

print(f"\nSelected {n_top} markers by mean rank.")
print(f"Top 5: {', '.join(hvg_df.head(5)['marker'])}")
print(f"Borderline (around cutoff): {', '.join(hvg_df.iloc[n_top-2:n_top+3]['marker'])}")

In [ ]:
# Apply HVM filter: keep top n_top markers and filter colocalization pairs
selected_markers = set(hvg_df.head(n_top)['marker'])

coloc_key = next((k for k in ['spatial_asinh5_top500_var', 'HOTSPOT_top500_var', 'HOTSPOT'] if k in cd8.obsm), None)
coloc_df = pd.DataFrame(cd8.obsm[coloc_key], index=cd8.obs_names)
pair_cols = [c for c in coloc_df.columns if '/' in str(c)]

# Keep only pairs where BOTH markers are in the selected set
filtered_cols = [c for c in pair_cols
                 if c.split('/')[0] in selected_markers and c.split('/')[1] in selected_markers]

filtered_key = f'{coloc_key}_hvm'
cd8.obsm[filtered_key] = coloc_df[filtered_cols]

print(f"Selected {len(selected_markers)} markers (top {n_top} by mean rank)")
print(f"Colocalization pairs retained: {len(filtered_cols)} / {len(pair_cols)}")
print(f"Filtered obsm key: '{filtered_key}'")

In [ ]:
cd8_PHA=cd8[cd8.obs['condition']=='PHA']
cd8_unstim=cd8[cd8.obs['condition']=='unstim']
G, partition = run_marker_community_analysis(cd8_PHA)


In [ ]:
import networkx as nx
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from community import community_louvain

def compare_marker_networks(adata, conditions=('unstim', 'PHA'), coloc_key=None, threshold=None, resolution=1.0):
    """
    All-in-one function to compare marker clustering between two conditions.
    1. Builds networks for both conditions using a shared threshold.
    2. Plots them side-by-side with a stable layout.
    3. Plots sorted correlation heatmaps for each condition.
    4. Plots a difference heatmap (condition2 - condition1) to show co-expression changes.
    5. Plots a Jaccard heatmap of cluster similarity.
    6. Prints the specific markers in each cluster.
    """
    
    # 1. Setup & Global Thresholding
    if not coloc_key:
        priority = ['spatial_asinh5_top500_var', 'HOTSPOT_top500_var', 'HOTSPOT']
        coloc_key = next((k for k in priority if k in adata.obsm), None)
        
    if not coloc_key:
        print("Error: No colocalization data found in adata.obsm"); return

    # Calculate global threshold across ALL data
    full_df = pd.DataFrame(adata.obsm[coloc_key], index=adata.obs_names)
    pair_cols = [c for c in full_df.columns if '/' in str(c)]
    all_weights = full_df[pair_cols].abs().mean()
    
    if threshold is None:
        threshold = max(all_weights.quantile(0.75), 0.01)
    
    print(f"Using Key: {coloc_key}")
    print(f"Global Threshold: {threshold:.4f} (Top 25% of all edges)")

    # 2. Build Graphs & Detect Communities
    results = {}
    graphs = {}
    
    for cond in conditions:
        cells = adata.obs[adata.obs['condition'] == cond].index
        if len(cells) == 0:
            print(f"Warning: No cells found for condition '{cond}'"); continue
            
        sub_df = pd.DataFrame(adata[cells].obsm[coloc_key], index=cells)
        cond_weights = sub_df[pair_cols].abs().mean()
        
        G = nx.Graph()
        for pair, w in cond_weights.items():
            if w >= threshold:
                m1, m2 = pair.split('/')
                if m1 != m2: G.add_edge(m1, m2, weight=w)
        
        if G.number_of_nodes() > 0:
            partition = community_louvain.best_partition(G, resolution=resolution, random_state=42)
            communities = {}
            for node, cid in partition.items():
                communities.setdefault(cid, []).append(node)
            results[cond] = communities
            graphs[cond] = {'G': G, 'partition': partition, 'weights': cond_weights}
        else:
            results[cond] = {}
            graphs[cond] = {'G': G, 'partition': {}, 'weights': cond_weights}

    # 3. Visualization A: Side-by-Side Networks (Fixed Layout)
    all_nodes = set().union(*[g['G'].nodes() for g in graphs.values()])
    G_union = nx.Graph(); G_union.add_nodes_from(all_nodes)
    for g in graphs.values(): G_union.add_edges_from(g['G'].edges())
    fixed_pos = nx.spring_layout(G_union, k=1.5, seed=42)

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    for idx, cond in enumerate(conditions):
        ax = axes[idx]
        data = graphs.get(cond)
        if data and data['G'].number_of_nodes() > 0:
            G = data['G']
            part = data['partition']
            local_pos = {n: fixed_pos[n] for n in G.nodes()}
            colors = [part.get(n, 0) for n in G.nodes()]
            edge_w = [G[u][v]['weight'] * 3 for u, v in G.edges()]
            
            nx.draw_networkx_edges(G, local_pos, alpha=0.2, width=edge_w, edge_color='gray', ax=ax)
            nx.draw_networkx_nodes(G, local_pos, node_color=colors, cmap=plt.cm.Set3, node_size=500, ax=ax)
            nx.draw_networkx_labels(G, local_pos, font_size=8, font_weight='bold', ax=ax)
            ax.set_title(f"{cond} Network\n({G.number_of_edges()} edges)")
            ax.axis('off')
        else:
            ax.text(0.5, 0.5, "No significant edges", ha='center'); ax.axis('off')
    plt.tight_layout(); plt.show()

    # 4. Visualization B: Marker Correlation Heatmaps (Sorted by Community)
    # Use union of markers from both conditions for shared ordering
    shared_markers = sorted(all_nodes)
    corr_matrices = {}
    
    fig, axes = plt.subplots(1, 2, figsize=(24, 10))
    
    for idx, cond in enumerate(conditions):
        ax = axes[idx]
        data = graphs.get(cond)
        
        if data and data['G'].number_of_nodes() > 0:
            G = data['G']
            partition = data['partition']
            weights = data['weights']
            
            # Sort nodes by community ID
            order = sorted(G.nodes(), key=lambda x: (partition[x], x))
            
            # Build correlation matrix on this condition's nodes
            corr_matrix = pd.DataFrame(0.0, index=order, columns=order)
            for pair, w in weights.items():
                m1, m2 = pair.split('/')
                if m1 in order and m2 in order:
                    corr_matrix.loc[m1, m2] = corr_matrix.loc[m2, m1] = w
            
            sns.heatmap(corr_matrix, cmap='RdBu_r', center=0, ax=ax, square=True, cbar_kws={"shrink": .5})
            ax.set_title(f"{cond}: Marker Correlation (Sorted by Community)")
            
            # Also build full matrix on shared markers for the difference heatmap
            full_corr = pd.DataFrame(0.0, index=shared_markers, columns=shared_markers)
            for pair, w in weights.items():
                m1, m2 = pair.split('/')
                if m1 in shared_markers and m2 in shared_markers:
                    full_corr.loc[m1, m2] = full_corr.loc[m2, m1] = w
            corr_matrices[cond] = full_corr
        else:
            ax.text(0.5, 0.5, "No Data", ha='center'); ax.axis('off')
            corr_matrices[cond] = pd.DataFrame(0.0, index=shared_markers, columns=shared_markers)

    plt.tight_layout(); plt.show()

    # 4b. Visualization B2: Difference Heatmap (condition2 - condition1)
    if len(corr_matrices) == 2:
        diff_matrix = corr_matrices[conditions[1]] - corr_matrices[conditions[0]]
        
        # Sort by the second condition's communities (the "effect" condition)
        if graphs.get(conditions[1], {}).get('partition'):
            part2 = graphs[conditions[1]]['partition']
            # Markers not in condition2's graph get sorted last
            diff_order = sorted(shared_markers, key=lambda x: (part2.get(x, 999), x))
        else:
            diff_order = shared_markers
        
        diff_matrix = diff_matrix.loc[diff_order, diff_order]
        
        # Symmetric color scale
        vmax = diff_matrix.abs().values.max()
        
        plt.figure(figsize=(14, 12))
        sns.heatmap(diff_matrix, cmap='RdBu_r', center=0, vmin=-vmax, vmax=vmax,
                    square=True, cbar_kws={"shrink": .5, "label": "Colocalization Change"})
        plt.title(f"Co-expression Change: {conditions[1]} - {conditions[0]}\n(Red = gained, Blue = lost)")
        plt.tight_layout(); plt.show()
        
        # Print top changed pairs
        pairs_changed = []
        for i, m1 in enumerate(diff_order):
            for j, m2 in enumerate(diff_order):
                if i < j and diff_matrix.loc[m1, m2] != 0:
                    pairs_changed.append({'Pair': f'{m1}/{m2}', 'Change': diff_matrix.loc[m1, m2]})
        if pairs_changed:
            df_changes = pd.DataFrame(pairs_changed).sort_values('Change', key=abs, ascending=False)
            print(f"\nTop 10 GAINED co-expression ({conditions[1]} vs {conditions[0]}):")
            display(df_changes.nlargest(10, 'Change'))
            print(f"\nTop 10 LOST co-expression ({conditions[1]} vs {conditions[0]}):")
            display(df_changes.nsmallest(10, 'Change'))

    # 5. Visualization C: Jaccard Similarity Heatmap
    c1, c2 = results.get(conditions[0], {}), results.get(conditions[1], {})
    if c1 and c2:
        mat = pd.DataFrame(index=[f"{conditions[0]}-{k}" for k in c1],
                           columns=[f"{conditions[1]}-{k}" for k in c2]).fillna(0.0)
        annot = pd.DataFrame(index=mat.index, columns=mat.columns)
        for k1, m1 in c1.items():
            for k2, m2 in c2.items():
                s1, s2 = set(m1), set(m2)
                overlap = len(s1 & s2)
                mat.iloc[k1, k2] = overlap / len(s1 | s2) if (s1 | s2) else 0
                annot.iloc[k1, k2] = str(overlap) if overlap > 0 else ""
        
        plt.figure(figsize=(8, 6))
        sns.heatmap(mat, annot=annot, fmt='', cmap='Blues', cbar_kws={'label': 'Jaccard Similarity'})
        plt.title(f"Cluster Reorganization: {conditions[0]} -> {conditions[1]}"); plt.show()

    # 6. Print Clusters
    print(f"\n{'='*60}\nDETAILED CLUSTER MEMBERSHIP\n{'='*60}")
    for cond in conditions:
        print(f"\n--- {cond.upper()} CONDITION ---")
        for cid, markers in sorted(results.get(cond, {}).items(), key=lambda x: -len(x[1])):
            print(f"Cluster {cid} ({len(markers)}): {', '.join(sorted(markers))}")
def analyze_centrality_shift(adata, conditions=('unstim', 'PHA'), coloc_key=None, threshold=None):
    print(f"\n{'='*80}\n3. CENTRALITY SHIFT ANALYSIS\n{'='*80}")
    
    if not coloc_key: coloc_key = next((k for k in ['spatial_asinh5_top500_var', 'HOTSPOT_top500_var'] if k in adata.obsm), None)
    
    # Build Graphs (Recalculate to ensure self-contained logic)
    full_df = pd.DataFrame(adata.obsm[coloc_key], index=adata.obs_names)
    pair_cols = [c for c in full_df.columns if '/' in str(c)]
    if threshold is None: threshold = max(full_df[pair_cols].abs().mean().quantile(0.75), 0.01)

    cent_data = {}
    for cond in conditions:
        cells = adata.obs[adata.obs['condition'] == cond].index
        sub_df = pd.DataFrame(adata[cells].obsm[coloc_key], index=cells)
        weights = sub_df[pair_cols].abs().mean()
        G = nx.Graph()
        for pair, w in weights.items():
            if w >= threshold:
                m1, m2 = pair.split('/')
                if m1 != m2: G.add_edge(m1, m2, weight=w)
        cent_data[cond] = nx.degree_centrality(G)

    # Compare
    all_nodes = set(cent_data[conditions[0]].keys()) | set(cent_data[conditions[1]].keys())
    df = pd.DataFrame([{
        'Marker': n, 
        conditions[0]: cent_data[conditions[0]].get(n, 0), 
        conditions[1]: cent_data[conditions[1]].get(n, 0)
    } for n in all_nodes])
    df['Shift'] = df[conditions[1]] - df[conditions[0]]
    
    plt.figure(figsize=(8, 8))
    sns.scatterplot(data=df, x=conditions[0], y=conditions[1], hue='Shift', palette='vlag', s=100)
    
    for _, r in pd.concat([df.nlargest(5, 'Shift'), df.nsmallest(5, 'Shift')]).iterrows():
        plt.text(r[conditions[0]], r[conditions[1]], r['Marker'], fontsize=9, weight='bold')
        
    plt.plot([0, df[[conditions[0], conditions[1]]].max().max()], [0, df[[conditions[0], conditions[1]]].max().max()], 'k--', alpha=0.3)
    plt.title(f"Centrality Shift: {conditions[0]} -> {conditions[1]}"); plt.show()
    
    print("Top 5 Gaining Importance:"); display(df.nlargest(5, 'Shift')[['Marker', 'Shift']])

# Run with filtered key (only abundant proteins)
compare_marker_networks(cd8, coloc_key=filtered_key, threshold=0.01, resolution=1.0)
analyze_centrality_shift(cd8, coloc_key=filtered_key)

In [ ]:
def extract_pha_induced_clusters(adata, conditions=('unstim', 'PHA'), coloc_key=None, threshold=None, resolution=1.0):
    """
    Extract clusters formed specifically from PHA stimulation by:
    1. Building the difference matrix (PHA - unstim co-expression)
    2. Running community detection on ONLY the positive edges (gained co-expression)
    3. Visualizing these PHA-induced modules as a network and heatmap
    """
    
    # --- Setup (same as compare_marker_networks) ---
    if not coloc_key:
        priority = ['spatial_asinh5_top500_var', 'HOTSPOT_top500_var', 'HOTSPOT']
        coloc_key = next((k for k in priority if k in adata.obsm), None)
    
    full_df = pd.DataFrame(adata.obsm[coloc_key], index=adata.obs_names)
    pair_cols = [c for c in full_df.columns if '/' in str(c)]
    
    if threshold is None:
        threshold = max(full_df[pair_cols].abs().mean().quantile(0.75), 0.01)
    
    # --- Build per-condition weight vectors ---
    cond_weights = {}
    for cond in conditions:
        cells = adata.obs[adata.obs['condition'] == cond].index
        sub_df = pd.DataFrame(adata[cells].obsm[coloc_key], index=cells)
        cond_weights[cond] = sub_df[pair_cols].abs().mean()
    
    # --- Compute difference (PHA - unstim) per pair ---
    diff_weights = cond_weights[conditions[1]] - cond_weights[conditions[0]]
    
    # --- Build graph from ONLY positive differences (gained co-expression) ---
    # Use threshold on the gain magnitude to keep meaningful edges
    gain_threshold = diff_weights[diff_weights > 0].quantile(0.5)  # top 50% of gains
    
    G_gained = nx.Graph()
    for pair, dw in diff_weights.items():
        if dw > gain_threshold:
            m1, m2 = pair.split('/')
            if m1 != m2:
                G_gained.add_edge(m1, m2, weight=dw)
    
    if G_gained.number_of_nodes() == 0:
        print("No significant gained co-expression edges found."); return
    
    # --- Community detection on gained edges ---
    partition = community_louvain.best_partition(G_gained, resolution=resolution, random_state=42)
    communities = {}
    for node, cid in partition.items():
        communities.setdefault(cid, []).append(node)
    
    # --- Visualization 1: PHA-induced network ---
    pos = nx.spring_layout(G_gained, k=1.5, seed=42)
    colors = [partition[n] for n in G_gained.nodes()]
    edge_w = [G_gained[u][v]['weight'] * 5 for u, v in G_gained.edges()]
    
    plt.figure(figsize=(12, 10))
    nx.draw_networkx_edges(G_gained, pos, alpha=0.3, width=edge_w, edge_color='gray')
    nx.draw_networkx_nodes(G_gained, pos, node_color=colors, cmap=plt.cm.Set2, node_size=600, edgecolors='black', linewidths=0.5)
    nx.draw_networkx_labels(G_gained, pos, font_size=9, font_weight='bold')
    plt.title(f"PHA-Induced Co-expression Clusters\n(Edges = gained colocalization, {G_gained.number_of_edges()} edges)")
    plt.axis('off'); plt.tight_layout(); plt.show()
    
    # --- Visualization 2: Difference heatmap sorted by PHA-induced communities ---
    all_markers = sorted(G_gained.nodes(), key=lambda x: (partition[x], x))
    
    diff_corr = pd.DataFrame(0.0, index=all_markers, columns=all_markers)
    for pair, dw in diff_weights.items():
        m1, m2 = pair.split('/')
        if m1 in all_markers and m2 in all_markers:
            diff_corr.loc[m1, m2] = diff_corr.loc[m2, m1] = dw
    
    vmax = diff_corr.abs().values.max()
    
    plt.figure(figsize=(14, 12))
    sns.heatmap(diff_corr, cmap='RdBu_r', center=0, vmin=-vmax, vmax=vmax,
                square=True, cbar_kws={"shrink": .5, "label": f"{conditions[1]} - {conditions[0]}"})
    plt.title(f"PHA-Induced Clusters: Co-expression Difference\n(Sorted by gained-community membership)")
    plt.tight_layout(); plt.show()
    
    # --- Print PHA-induced clusters ---
    print(f"\n{'='*60}")
    print(f"PHA-INDUCED CO-EXPRESSION CLUSTERS")
    print(f"(Communities detected on gained edges only)")
    print(f"{'='*60}")
    for cid, markers in sorted(communities.items(), key=lambda x: -len(x[1])):
        # For each cluster, show which edges were gained
        intra_edges = []
        for u, v, d in G_gained.edges(data=True):
            if partition.get(u) == cid and partition.get(v) == cid:
                intra_edges.append((u, v, d['weight']))
        intra_edges.sort(key=lambda x: -x[2])
        
        print(f"\nCluster {cid} ({len(markers)} markers): {', '.join(sorted(markers))}")
        if intra_edges:
            print(f"  Top gained edges:")
            for u, v, w in intra_edges[:5]:
                print(f"    {u} -- {v}  (gain: {w:.4f})")
    
    return communities, G_gained, partition

# Run it
pha_clusters, G_pha, pha_partition = extract_pha_induced_clusters(cd8, coloc_key=filtered_key)